# 3-1: Text Analysis Continued

## Fast Review

1. What is text preprocessing?
2. Why might we lowercase text before counting words?
3. What does it mean to tokenize a text?
4. What are stop words?
5. What is a dataframe row?
6. Why might counting words in several texts be different from counting words in one text?


In our last tutorial, we studied one novel: Bram Stoker's _Dracula_. In this notebook, we'll extend those same ideas to a small __corpus__: six novels by Jane Austen. In DH, we often describe a dataset of texts as a corpus, meaning "body" of work. A corpus is just another word for dataset, but it usually denotes a dataset of an author's works, a genre, a period, etc. In other words, it usually means a curated dataset, defined by a certain subject matter.

Working with more than one text in a corpus changes the questions we can ask. Instead of only asking, "What words are common in this novel?" we can start asking comparative questions: Which words are distinctive to one novel? Which character names cluster together? How much quoted speech appears in each text? When you're doing DH work, you need to consider the kinds of questions you can reasonably ask of your data. To know what questions are reasonable, you should have a good sense of the texts in your data and how they were curated.

In this tutorial, we'll use `jane_austen_texts.csv`, saved locally in the `data` folder. Each row in the csv is one Jane Austen novel (six total). What kinds of questions could we ask of this dataset?

## Learning Objectives

By the end of this notebook, you should be able to:

1. Review common preprocessing steps: lowercasing, punctuation removal, tokenization, and stop word removal.
2. Iterate over rows in a pandas dataframe.
3. Use regular expressions to extract quoted dialogue from texts.
4. Understand, create, and count n-grams.
5. Use part-of-speech (pos) tagging to identify broad grammatical patterns.
6. Use TF-IDF to identify words that are especially distinctive within a small corpus.
7. Explain which methods are rule-based, count-based, or built from machine learning.

## Text Analysis Libraries

We'll use some libraries from last time and add a few new tools:

- `string`: gives us a ready-made list of common punctuation characters.
- `re`: Python's regular expressions module.
- `pandas`: our tried-and-true library for working with tabular data.
- `matplotlib`: helps us make quick plots.
- `nltk`: provides tokenization, frequency counts, stop words, n-grams, and pos tagging.
- `scikit-learn`: a popular machine learning library that provides TF-IDF tools through `TfidfVectorizer`. TF-IDF stands for "Term Frequency-Inverse Document Frequency". More info to follow.

At this point in our text analysis tutorials, we're touching on some __machine learning__ methods. In particular, pos tagging. But not everything in text analysis is machine learning. Preprocessing, regular expressions, and frequency counts are mostly __rule-based__ or __count-based__ methods. POS tagging uses a __trained statistical/machine-learning model__. TF-IDF isn't a predictive model by itself, but it is a corpus-based weighting method that is often used to turn texts into features for machine learning.

Let's import what we need:

In [ ]:
from string import punctuation
import re

import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.util import ngrams

from sklearn.feature_extraction.text import TfidfVectorizer

Some `nltk` tools use language data that may need to be downloaded once on your computer. The cell below downloads the resources used in this notebook.


In [ ]:
# nltk's tokenizer model
nltk.download("punkt", quiet=True)
# supplementary to nltk's tokenizer model
# adds more info to model regarding word boundaries
nltk.download("punkt_tab", quiet=True)
# a list of common stopwords
nltk.download("stopwords", quiet=True)
# nltk's pretrained pos tagging model
nltk.download("averaged_perceptron_tagger", quiet=True)
# English-specific version of the pos tagging model
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

## Loading the Corpus

Alright, let's load our corpus into the session. Below we're doing that by pointing pandas to the file using the relative path: `../data/jane_austen_texts.csv`. Remember, that means: "go up one folder, then go into the `data` folder."


In [ ]:
austen_texts = pd.read_csv("../data/jane_austen_texts.csv")

`austen_texts' is a pandas dataframe. How can we review it's general shape and contents quickly?

In [ ]:
print("No. of rows:", len(austen_texts))
austen_texts.head()

What does that tell us?

We have six rows (a row per novel). We have titles, years published, author, and the full text of each novel. What kinds of data types are these?

Let's inspect the metadata columns first. We do not want to print the full texts yet because each one is a whole novel.


In [ ]:
austen_texts.dtypes

Looks like we've got strings and one column with integers. It's always good to double-check before you run more code on the dataframe.

How can we review the contents of the first three columns?

In [ ]:
# remember list within a list structure
austen_texts[["title", "year_published", "author"]]

All right, and then we've got that other column `full_text`. If true, that column will have some long strings in it!

Let's check and save the results in a new column called `string_length`. We can do that like this:

It is also useful to check the size of each text before doing analysis. This gives us a quick sense of the corpus.


In [ ]:
# remember the .str method (reads the data in assign column as a string)
austen_texts["string_length"] = austen_texts["full_text"].str.len()
austen_texts.head()

That's a lot of characters! Seems to be true that we're dealing with book-length texts in the `full_text` column.

How about if we get a rough estimate of the number of words in each `full_text` row?

In [ ]:
# split method designates words on whitespace
austen_texts["rough_word_count"] = austen_texts["full_text"].str.split().str.len()
austen_texts.head()

## Adding Rows to Dataframes

But wait! There's one more text we might want to add to our dataframe: `lady_susan.txt`. It's in the data folder, too. It's a posthumously published Jane Austen novella. But how can we add it to our dataframe?

First let's get the create a path to the file using `open()`. Remember: `open()` doesn't "read" the file, it just "opens the door" to the file.

In [ ]:
lady_susan_file = open("../data/lady_susan.txt", "r", encoding="utf-8")

Now we can use `read()` to get the text of Lady Susan. To fill in the other column values for this new row, we'll also create a tiny dataframe for it, assigning the column values with a little dictionary, like this:

In [ ]:
lady_susan_text = lady_susan_file.read()

new_row = pd.DataFrame([{
    "title": "Lady Susan",
    "year_published": 1871,
    "author": "Jane Austen",
    "full_text": lady_susan_text
}])

All right, let's see what we've created:

In [ ]:
new_row

It's just a tiny baby dataframe. But importantly, its columns match our austen_texts dataframe. To combine them, we just need to use the `.concat()` pandas method, like this:

In [ ]:
# the ignore_index argument ensures we don't have conflicting indices
austen_texts = pd.concat([austen_texts, new_row], ignore_index=True)
austen_texts

That looks pretty good! But notice how the Lady Susan row got the columns we created earlier (string_length and rough_word_count) without values, just NaN. We'll need to rerun our code to fill in those values:

In [ ]:
# remember the .str method (reads the data in assign column as a string)
austen_texts["string_length"] = austen_texts["full_text"].str.len()
# split method designates words on whitespace
austen_texts["rough_word_count"] = austen_texts["full_text"].str.split().str.len()
austen_texts

## Preprocessing Recap

We have a good little dataframe now. Notice how the full_text values don't contain any Gutenberg boilerplate. That's already been removed, but it's still good to double-check sometimes. You never know what strange artifacts might be in your textual data.

If you wanted to double-check, you could use `.loc` to read the row values at the beginnings and the ends. For example:

In [ ]:
# first 100 characters of first row's "full_text" string
print(austen_texts["full_text"].loc[0][:100])

In [ ]:
# last 100 characters of first row's "full_text" string
print(austen_texts["full_text"].loc[0][-100:])

So that's good, that little preprocessing step seems to be taken care of. But let's recap quickly the other common preprocessing steps we covered last time:

- lowercasing the text
- removing punctuation
- tokenizing the text
- removing stop words

Remember, there's no one correct preprocessing workflow. The right workflow depends on the question. If you care about dialogue, you probably should not remove quotation marks before extracting dialogue. If you care about sentence style, you may need to keep punctuation. If you care about word frequencies, lowercasing and punctuation removal are often useful.

In our case, we're gonna do some interrogation of n-grams (sequences of words), dialogue extraction, pos tagging, and TF-IDF. In these cases, it'll be good to lowercase and remove _most_ punctuation, but not necessarily _all_ punctuation, and eventually remove stopwords. But that's all to say that you should choose your preprocessing based on your methods and research questions. It will make a world of difference.

### A Small Sample

And also, it's always good to create a new object––a sample––to futz with your preprocessing steps before you deploy them on your full dataset. In this tutorial, we're still working with a small dataset. DH researchers often do things with hundreds or thousands of textual examples, though. Before running code on large data, it's good to make sure it works on a sample first!

So, let's test our preprocessing steps on a small passage from _Emma_. We can get the sample like this:

In [ ]:
# object emma_row is assigned to row where "Title" is equal to "Emma"
emma_row = austen_texts[austen_texts["title"] == "Emma"]
# then selects emma_row "full_text" value to emma_text
emma_text = emma_row["full_text"].iloc[0]

# finds string "Emma Woodhouse" in emma_text
sample_start = emma_text.find("Emma Woodhouse")
# creates sample of sample_start + 5000 next characters
sample = emma_text[sample_start:sample_start + 5000]

print(sample)

### Lowercasing

Okay, for our purposes, we'll want to lowercase our text so instances like `Emma`, `emma`, and `EMMA` all get counted as the same word. Lowercasing, you may remember, is real easy. We've got a method for it:


In [ ]:
sample_lower = sample.lower()

print(sample_lower[:500])

### Removing Punctuation

Removing punctuation is made easier with the `punctuation` variable from Python's `string` module. It contains many common punctuation characters. They are:


In [ ]:
print(punctuation)

Let's use it now to remove common punctuation and check the results on our sample:

In [ ]:
sample_clean = sample_lower

# for loop to loop over examples in punctuation
for character in punctuation:
    # applies each character with .replace()
    sample_clean = sample_clean.replace(character, " ")

print(sample_clean)

Hmm, look carefully at that pass. Notice anything?

There are still `—` characters. This little bit of punctuation isn't part of the `string` object `punctuation`. We need to add it to our preprocessing ourselves.

To do that, we can create our own object `extra_punctuation`. It's got the `—` and some other stuff I've identified by looking over these Jane Austen texts more closely by hand. We just need to add `extra_punctuation` to `punctuation` and run it again:

In [ ]:
extra_punctuation = "“”‘’—–"
all_punctuation = punctuation + extra_punctuation

sample_clean = sample_lower

for character in all_punctuation:
    sample_clean = sample_clean.replace(character, " ")

print(sample_clean)

### Tokenization

Okay, with our punctuation removal process established, we're ready to tokenize our text. __Tokenization__, you may remember, means splitting a text into smaller pieces called tokens. These are often words, but tokens can also be sentences, clauses, paragraphs––any unit of the text you decide.

Since we've already lowercased and removed punctuation, a simple `.split()` gives us a reasonable first list of word tokens:

In [ ]:
# split sets words by whitespace
sample_tokens = sample_clean.split()

print(sample_tokens[:80])
print("Number of sample tokens:", len(sample_tokens))

### Stop Word Removal

Finally, we're ready to establish our preprocessing for the elimination of stop words. Stop words, you may remember, are common words like `the`, `and`, `to`, and `of` (articles, prepositions, pronouns, etc.) They can be meaningful, but they often dominate frequency counts. Removing them can help us see more content-heavy words.

Remember that `nltk` has a stopwords list in English (`stopwords.words("english)`). We'll use it to remove stop words. To do that, we'll create the __accumulator variable__ `sample_no_stops`. Then we'll iterate over our `sample_tokens` and if a token isn't in `stop_words`, we'll add it to our accumulator:

In [ ]:
stop_words = set(stopwords.words("english"))

# the accumulator shell, an empty list
sample_no_stops = []

for token in sample_tokens:
    if token not in stop_words:
        #.append() to add tokens to accumulator
        sample_no_stops.append(token)

print(sample_no_stops[:80])
print("Tokens before stop word removal:", len(sample_tokens))
print("Tokens after stop word removal:", len(sample_no_stops))

### A Reusable Preprocessing Function

All right, we've got the business now! All our preprocessing seems to have worked on our sample. We're ready to apply it to the rest of our data.

Because we'll repeat these steps many times, let's wrap them in a function. This is a textbook example of a time when you'd want to write your own function.

We'll call it `clean_and_tokenize()`. It will take an object `text`, then:

- lowercase the text
- replace punctuation with blank space
- split the text into tokens with `.split()`

Remember the structure of functions? You name the function with `def` then add its steps as indented lines of code. At the end, you must use `return` so the function outputs your chosen variable.

In [ ]:
def clean_and_tokenize(text):
    """
    takes text and returns it lowercased, punctuation removed, and tokenized by whitespace.
    """
    clean_text = text.lower()

    for character in all_punctuation:
        clean_text = clean_text.replace(character, " ")

    tokens = clean_text.split()

    return tokens

In [ ]:
help(clean_and_tokenize)

Now let's run it on the full text of Emma:

In [ ]:
# our new clean_and_tokenize() function in action
emma_tokens = clean_and_tokenize(emma_text)

# let's compare before and after
print(emma_text[:10])
print("Total tokens (characters) in preprocessed Emma:", len(emma_text))
print("Unique tokens (characters) in preprocessed Emma:", len(set(emma_text)))
print("---")
print(emma_tokens[:10])
print("Total tokens (words) in processed Emma:", len(emma_tokens))
print("Unique tokens (words) in processed Emma:", len(set(emma_tokens)))

### Iterating Over Rows in a Dataframe

Each row in `austen_texts` is one novel. When we want to run the same analysis or transformation on every text, we can loop over the rows.

The `.iterrows()` method gives us two things each time through the loop:

- the row's index
- the row's values as a pandas Series

Let's try iterating over austen_texts with `.iterrows()`. In this example, we'll assign objects to each row value then print those objects in a little summary report of the data:

In [ ]:
# notice this for loop assigns two loop variables
for index, row in austen_texts.iterrows():
    title = row["title"]
    year = row["year_published"]
    text_length = len(row["full_text"])

    print(index, title, year, text_length)

Just a brief refresher––what's going on with those loop variables `index` and `row`? What are they?

In [ ]:
print("Last index counted:", index)
print("Last row counted:", row)

### Applying Functions to Columns in a Dataframe

One other nice aspect of pandas is the `.apply()` method. This method takes a function and applies it to all the data in a given column. In our case, we can take our `clean_and_tokenize` function and deploy it so every row in our data gets a cleaned version.

Let's do that and save the results in a new column called `"cleaned_text"`. We'll also count the words in `"cleaned_text"` and save the result in a column called `"word_count"`:

In [ ]:
# notice the .apply() method takes a function as its argument
austen_texts["cleaned_text"] = austen_texts["full_text"].apply(clean_and_tokenize)
austen_texts["word_count"] = austen_texts["cleaned_text"].apply(len)
austen_texts.head()

Neat! Now we can use the dataframe `austen_texts` to create some visualizations of our cleaned texts.

Let's use the "word_count" column values to create a barplot comparing the lengths of the novels in our data:

In [ ]:
austen_texts.plot(kind="bar", x="title", y="word_count", legend=False)

plt.title("Total Words in Jane Austen Novels")
plt.xlabel("Novel")
plt.ylabel("Words")
# notice .xticks method rotates ticks to make them readable
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Regular Expressions for Dialogue

We went over regular expressions, often called regex, briefly in `2-4_regular_expressions.ipynb`. But we haven't really seen a "real-world" DH example of when we might use them. Let's do that now. 

Remember: regular expressions let us search for patterns instead of exact words. For example, if we wanted to extract quoted dialogue, we're not looking for one specific phrase. We're looking for a pattern: an opening quotation mark, then some text, then a closing quotation mark.

Of course, this only works if a text consistently uses quotation marks to signal dialogue! That's not always the case, but presuming our corpus has quoted dialogue, it's one way we can use regular expressions.

This example showcases how regex is both powerful and interpretable, but ultimately a __rule-based method__. They require the text follows certain pattern rules. Otherwise, they're brittle. The method breaks or doesn't work properly.

Let's try it on our Jane Austen corpus. But first, one small complication: these Gutenberg files use curly quotation marks (`“` and `”`) rather than straight quotation marks (`"`). How do I know this? By (human) reading some of the data. This is an important practice––one I conducted as I put together this tutorial. If something with regex isn't working as expected, be sure to read examples yourself!

Anyway, we can normalize the curly marks to straight marks, then write a regex pattern around `"`. That's fairly easy. Let's look at the quotation mark types by count in our `emma_text` variable (the version prior to punctuation removal):

In [ ]:
print('Straight double quote characters:', emma_text.count('"'))
print("Curly opening quote characters:", emma_text.count("“"))
print("Curly closing quote characters:", emma_text.count("”"))

That's a lot of quotations! Over 2,000 of them. To extract them using regular expressions, we need to first normalize the quotation marks to the standard `"`. Then we need to build a regex pattern that captures:

1) the first quotation mark
2) all the text within the quotation mark
3) the ending quotation mark

The first and third parts are easy. We after normalizing the quotations, we just use `"` in our regex pattern. The middle part, however, is more complicated. How do we translate the pattern of any/all characters between quotations?

To figure that out, you may want to toy with things in [https://regex101.com/](https://regex101.com/). It's also a good thing to ask an LLM about. But be sure you've tested your regex patterns and that you understand them at each symbol/step before analyzing the results.

I've put together a simple pattern here. The pattern below means:

- `"` match an opening straight quotation mark
- `(` start a captured group
- `[^"]+` match one or more characters that are not quotation marks
- `)` end the captured group
- `"` match a closing straight quotation mark

Let's turn that into Python:

In [ ]:
# standardize the quotation marks
emma_for_regex = emma_text.replace("“", '"').replace("”", '"')
# save my regex pattern to object
dialogue_pattern = r'"([^"]+)"'

# use re.finall w/ dialogue_pattern to get dialiogue
emma_quoted_passages = re.findall(dialogue_pattern, emma_for_regex)

print("Quoted passages in Emma:", len(emma_quoted_passages))
print(emma_quoted_passages[:5])


That seem to have worked! We have the quoted passages of _Emma_ now, saved as a list object called `emma_quoted_passages`:

In [ ]:
type(emma_quoted_passages)

Now let's turn this dialogue extraction process into a function and apply it to every novel:

In [ ]:
def extract_quoted_passages(text):
    dialogue_pattern = r'"([^"]+)"'
    normalized_text = text.replace("“", '"').replace("”", '"')
    return re.findall(dialogue_pattern, normalized_text)

And we deploy with `.apply()`!

In [ ]:
austen_texts["dialogue"] = austen_texts["full_text"].apply(extract_quoted_passages)
austen_texts.head()

That's pretty neat. Now how can we use this data?

Well, let's say we wanted to compare novels by the amount of their dialogue. To do that, we can take the `dialogue` extracted lists, count their words, and then get their word counts against the total word counts of the novels.

To do that, remember that our `dialogue` column contains lists of strings:

In [ ]:
type(austen_texts["dialogue"][0])

That means we need to transform these lists into strings so we can count the words in the same way we counted words overall: with our `clean_and_tokenize()` function. So, in pseudocode, the process would be:

1) transform "dialogue" column values into strings
2) clean the dialogue strings (lowercase, remove punctuation, tokenize)
3) count the dialogue strings
4) divide numbers of dialogue strings to numbers of total words to get the "dialogue ratio"

Let's translate that to code now:

In [ ]:
# new column, dialogue read as strings
austen_texts["dialogue_word_count"] = austen_texts["dialogue"].astype(str)
# clean the dialogue strings
austen_texts["dialogue_word_count"] = austen_texts["dialogue_word_count"].apply(clean_and_tokenize)
# count the tokenized strings
austen_texts["dialogue_word_count"] = austen_texts["dialogue_word_count"].apply(len)
# calculate dialogue ratio
austen_texts["dialogue_ratio"] = austen_texts["dialogue_word_count"] / austen_texts["word_count"]
print(austen_texts[["title", "dialogue_ratio"]])

Cool! Now what do these results tell us?

Let's visualize these results using matplotlib:

In [ ]:
# notice the arguments passed into matplotlib: kind, x, y, legend
austen_texts.plot(kind="bar", x="title", y="dialogue_ratio", legend=False)

plt.title("Approximate Percentage of Dialogue in Austen Novels")
plt.xlabel("Novel")
plt.ylabel("Percentage of Words in Quotation Marks")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## N-Grams

Let's turn now to __n-grams__. An n-gram is a sequence of `n` tokens. It's a way to define units of text. We've looked at tokenizing by words, but you can also tokenize by a certain number of n-gram units. For example:

- A unigram is one token: `emma`.
- A bigram is two tokens in a row: `miss woodhouse`.
- A trigram is three tokens in a row: `poor miss taylor`.

N-grams are useful because they help us study repeated phrases, names, titles, and word pairings. A word frequency list can tell us that `miss` and `woodhouse` are common. A bigram count can tell us that `miss woodhouse` appears as a repeated phrase.

Let's start with a tiny example. You can quickly check a list of words for n-grams using the `nltk` function `ngrams()`:

In [ ]:
# list of tokens
tiny_tokens = ["it", "was", "a", "truth", "universally", "acknowledged",
               "it", "was", "a", "truth", "universally", "acknowledged"]

# ngram list:
print("Bigrams:", list(ngrams(tiny_tokens, 2)))
print("Trigrams:", list(ngrams(tiny_tokens, 3)))

print("No. of Bigrams as list:", len(list(ngrams(tiny_tokens, 2))))
print("No. of Trigrams as list:", len(list(ngrams(tiny_tokens, 3))))

# ngram set:
print("Bigrams:", set(ngrams(tiny_tokens, 2)))
print("Trigrams:", set(ngrams(tiny_tokens, 3)))

print("No. of Bigrams as set:", len(set(ngrams(tiny_tokens, 2))))
print("No. of Trigrams as set:", len(set(ngrams(tiny_tokens, 3))))

Notice how `ngrams()` creates overlapping tokens? It takes every _pairing_.

Notice also how ngram objects read as `list` versus `set` shows a different amount. Do you remember why? What's the difference between data structures `list` versus `set`?

But what's something we can do with n-grams? Well, how about we compile them and count them?

To do that, let's use the `austen_texts["cleaned_text"]` column values. We want to turn those values (lists of tokenized words) and turn them into trigrams. And count the trigrams, too, to see the most frequent ones. In this case, nltk's FreqDist object might be the best way to transform our text into trigrams. You may remember that FreqDist objects pair tokens with their frequency.

To do that, we'll first need to make the trigrams. There's a couple ways to do this. Again, we need to iterate over rows in our dataframe. We can do that with a for-loop that applies the cleaned_text to a list of ngrams and saves the results in a new column called "trigrams", like this:

In [ ]:
# our accumulator shell
austen_texts["trigrams"] = None

# for index and its accompanying row, iterate over them
for index, row in austen_texts.iterrows():
    # turn cleaned_text value into an object
    tokens = row["cleaned_text"]
    # use ngrams to turn tokens into trigram FreqDist
    trigrams = nltk.FreqDist(ngrams(tokens, 3))
    # put trigrams value into bigrams column
    austen_texts.at[index, "trigrams"] = trigrams

austen_texts.head()

What's another way to do that?

We could also create a function then use pandas `.apply()` to iterate over the rows. Here's what that might look like:

In [ ]:
# define the function and what it operates on (tokens)
def make_trigram_freqdist (tokens):
    # have it return a FreqDist object of trigrams
    return nltk.FreqDist(ngrams(tokens, 3))

Notice how that function takes the most critical part of our loop (`return nltk.FreqDist`)? Now with `.apply()`, we can run it on our dataframe:

In [ ]:
austen_texts["trigrams"] = austen_texts["cleaned_text"].apply(make_trigram_freqdist)
austen_texts.head()

It works the same!

But okay, let's take a look at some of these trigrams. In particular, let's look at the most frequent ones in _Pride and Prejudice_:

In [ ]:
top_trigrams_pride = austen_texts.loc[austen_texts["title"] == "Pride and Prejudice", 
                                    "trigrams"].iloc[0]

top_trigrams_pride = top_trigrams_pride.most_common(20)
top_trigrams_pride

What do these trigrams tell us about the text?

Perhaps we can investigate them with a visualization. That might make it easier to interpret them. Remember: pandas dataframes work cleanly with matplotlib, so let's first turn our `FreqDist` object `top_trigrams_pride` into a dataframe, like this:

In [ ]:
# notice how we're assigning column names
trigram_df = pd.DataFrame(top_trigrams_pride, columns=["trigram", "count"])
trigram_df

Now let's plug this dataframe into a visualization. We'll do a horizontal barplot with `.barh`:

In [ ]:
trigram_df.plot.barh(x="trigram", y="count", legend=False)

plt.xlabel("Frequency")
plt.ylabel("Trigram")
plt.title("Most Frequent Trigrams in Pride and Prejudice")
# .gca stands for "get current axis", invert_yaxis puts top bars on top
plt.gca().invert_yaxis()
plt.show()


What do you notice? 

Looks like there's some forms of address, some repeated copyright text, and other stuff. What do these tell you that single word frequencies couldn't?

## POS Tagging

Part-of-speech (POS) tagging tries to label each token with its grammatical role: noun, verb, adjective, adverb, and so on.

POS tagging is useful when your question is not only "which words appear?" but "what kinds of words appear?" For example, you might compare adjectives across novels, study verbs used around a character, or look for proper nouns as a rough way to find names.

This is our first clearly machine-learning-based method. Nltk's pos tagger has been trained on human-labeled text. When we give it a new sentence, it predicts the most likely tag for each word based on patterns it learned during training. This is an example of __supervised machine learning__. That is, the model was given examples and improved via human supervision. 

Pos tags are useful, but they can be wrong. Nltk's pos_tag() method was trained using the Penn Treebank schema on a corpus of mostly 20th century newspaper text. For some texts, it may not work as well because the patterns it learned from its training data don't transfer over.

POS tagging also works better when capitalization and punctuation are still available, so we'll have to use a different tokenizer for this section instead of our lowercased cleaning function.

Let's start there. Let's make a function that tokenizes while leaving capital letters  and punctuation intact. Just briefly, why do we need to leave capitalization and punctuation intact?

Consider capitalization––it often indicates proper nouns. Consider the word: "apple". When it's capitalized, it refers to the company Apple. Lowercased, it's a fruit. Duh! But not so obvious to our models.

Consider punctuation––a comma can make all the difference. "Let's eat, Grandma" and "Let's eat Grandma" mean ... erm... very different things.

So let's factor these things into our tokenizing function. In this instance, maybe we'll use nltk's built-in tokenizer. It's smarter than a simple `.split` and identifies words with more boundaries than mere whitespace factored in. Let's also ammend our punctuation list. We want to keep apostrophes and sentence indicators so our pos tagger can identify words like "Emma's" or "don't" as well as when words are capitalized because they're just at the start of a sentence. Here's how we can do it:

In [ ]:
pos_punctuation = all_punctuation.replace(".", "")
pos_punctuation = pos_punctuation.replace("?", "")
pos_punctuation = pos_punctuation.replace("!", "")
pos_punctuation = pos_punctuation.replace("'", "")
pos_punctuation = pos_punctuation.replace("’", "")

In [ ]:
def tokenize_for_pos(text):
    """
    removes some punctuation except apostrophes
    """
    for character in pos_punctuation:
        text = text.replace(character, " ")

    tokens = nltk.word_tokenize(text)
    return tokens

Let's tag a short passage from the beginning of _Emma_ first.


In [ ]:
# apply our tokenize_for_pos to our sample text
emma_pos_tokens = tokenize_for_pos(sample)
# use ntlk's pos_tag method to tag our sample
emma_pos_tags = nltk.pos_tag(emma_pos_tokens)
emma_pos_tags[0:20]


What do these tags mean, though? They are abbreviations for parts of speech. You can see [the key for them here.](https://gist.github.com/amnrzv/9a701f419ad004e066e2d6007dae40ad)

A few common POS tags:

- `NN`: singular noun.
- `NNS`: plural noun.
- `NNP`: proper noun.
- `JJ`: adjective.
- `RB`: adverb.
- `VB`, `VBD`, `VBG`, `VBN`, `VBP`, `VBZ`: verb forms.

Again, these tags are not likely to be perfect. They are predictions based on patterns in language. Still, they can help us ask questions about style and grammar at a scale that would be difficult by hand.

For example, what if we wanted to count pos tags in a text? We've just created a list of words and their parts-of-speech from our sample. To count the pos, all we need to do is parse the list, and accumulate the counts for each tag.

This is only a trickier step because it requires a loop with several if-else checks to add to the accumulations. But structurally, it's a straightforward process. In Python, it would look like this:


In [ ]:
# accumulator shell dictionary
counts = {
        "nouns": 0,
        "proper_nouns": 0,
        "adjectives": 0,
        "verbs": 0,
        "adverbs": 0
    }

# we iterate over both the word and its tag in our sample
for word, tag in emma_pos_tags:
    if tag in ["NN", "NNS"]:
        counts["nouns"] += 1
    elif tag in ["NNP", "NNPS"]:
        counts["proper_nouns"] += 1
    elif tag.startswith("JJ"):
        counts["adjectives"] += 1
    elif tag.startswith("VB"):
        counts["verbs"] += 1
    elif tag.startswith("RB"):
        counts["adverbs"] += 1

print(counts)

Pretty straightforward, right? Sometimes, big chunks of code are just that way. 

Now let's see if we can put that into a function. Presuming we want to iterate those steps over all our rows, it'll be best to make it a function:

In [ ]:
def count_broad_pos_categories(text):

    counts = {
        "nouns": 0,
        "proper_nouns": 0,
        "adjectives": 0,
        "verbs": 0,
        "adverbs": 0
    }

    for word, tag in text:
        if tag in ["NN", "NNS"]:
            counts["nouns"] += 1
        elif tag in ["NNP", "NNPS"]:
            counts["proper_nouns"] += 1
        elif tag.startswith("JJ"):
            counts["adjectives"] += 1
        elif tag.startswith("VB"):
            counts["verbs"] += 1
        elif tag.startswith("RB"):
            counts["adverbs"] += 1

    return counts


Almost forgot! We also need to make a little function to deploy the `nltk.pos_tag` to each row. Here's how we can do that:

In [ ]:
def pos_tag_text(text):
    return nltk.pos_tag(text)

Let's bring these functions together. First we'll use `tokenize_for_pos` to prepare our Austen texts for pos-tagging. Then we'll run the pos model on each row with `pos_tag_text`. Finally, we'll use `count_broad_pos_categories` to get our pos counts per novel:

In [ ]:
austen_texts["pos_tagged_text"] = austen_texts["full_text"].apply(tokenize_for_pos)

austen_texts["pos_tagged_text"] = austen_texts["pos_tagged_text"].apply(pos_tag_text)

austen_texts["pos_counts"] = austen_texts["pos_tagged_text"].apply(count_broad_pos_categories)

austen_texts.head()

That worked! But what exactly did we just create? Let's use `type()` to take a look:

In [ ]:
type(austen_texts["pos_counts"][0])

Dictionaries. We made a column with dictionaries. Each one looks like this:

In [ ]:
austen_texts["pos_counts"][0]

Let's visualize one of these dictionaries. We'll turn to _Pride and Prejudice_ again. Just like we did for n-grams, let's identify the row by title then turn its value into a dataframe with columns assigned:

In [ ]:
pos_counts_pride = austen_texts.loc[austen_texts["title"] == "Pride and Prejudice", 
                                    "pos_counts"].iloc[0]

pos_counts_pride_df = pd.DataFrame(pos_counts_pride.items(), 
                                   columns=["pos_category", "count"])

All right, now we just need to plug this dataframe into a matplotlib visualization, as we've done before:

In [ ]:
pos_counts_pride_df.plot.bar(x="pos_category", y="count", legend=False)

plt.xlabel("POS Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("POS Counts in Pride and Prejudice")
plt.show()


It worked! What could this tell us about the text? Presuming we've never read _Pride and Prejudice_, what simple things can we infer from these pos counts?

One last thing on pos counts: let's compare each novel by pos count. Since we've saved the pos counts as dictionaries, this is easy. All we have to do is use `.tolist()` to turn those column values into a Python list of dictionaries. Then pd.DataFrame transforms that list of dictionaries into a new dataframe!

In [ ]:
pos_counts_df = pd.DataFrame(austen_texts["pos_counts"].tolist())

pos_counts_df["title"] = austen_texts["title"]

pos_counts_df

In [ ]:
# notice how we can pass a list of column values to the y variable to create multiple bars
pos_counts_df.plot.bar(x="title", y=["nouns", "proper_nouns", "adjectives", "verbs", "adverbs"])

plt.xlabel("Text")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("POS Counts Across Jane Austen Texts")
plt.show()

## TF-IDF

TF-IDF stands for "term frequency-inverse document frequency." That's a mouthful, and admittedly confusing! But the basic idea is:

- __term frequency__: a word matters more if it appears often in a document
- __inverse document frequency__: a word matters less if it appears in many documents

In other words, TF-IDF is a measure of a word's _uniqueness_ in terms of its frequency within the given text. If A high TF-IDF score means a word appears often in the text but not as often across other texts.

In this notebook, you can think of each novel as one document. TF-IDF can help us identify words that are especially distinctive to one novel in this small Austen corpus. This is different from a plain frequency count. A word can be frequent in every novel and therefore not very distinctive.

TF-IDF is machine-learning-adjacent rather than a predictive machine-learning model by itself. The vectorizer `fits` itself to our corpus by learning the vocabulary and calculating IDF weights from the documents. Those weighted term scores are often used as input features for machine-learning models such as classifiers or clustering algorithms. Later on in this course, we'll circle back to TF-IDF for machine learning. For now, we're just using it to compare textual features across our corpus.

`scikit-learn` gives us a tool called `TfidfVectorizer`. It handles lowercasing, tokenization, stop word removal, and TF-IDF calculation all on its own. This `scikit-learn` is a new library for us. It's a popular machine-learning library with lots of cool features. `TfidfVectorizer` just scratches the surface.

The word `fit` becomes important here. It's got a distinct meaning in machine learning. When we call `.fit_transform()`, the vectorizer learns from this specific corpus: it records which terms appear and how widely they are distributed across the six novels. It's not learning literary meaning in a human sense, but it is learning a numerical representation from the data.

To start, we need to define our __vectorizer__. A vectorizer is a tool that converts text into a table where the rows are texts, the columns are words, and the values are numbers. In this case, those numbers are TF-IDF scores, which estimate how distinctive each word is in each text.

When we assign this type to the `vectorizer`, we're setting the parameters for `TfidfVectorizer()`. There are lots of parameters we could set, but for now, let's just do two:

- `stop_words`: the list of stopwords we want the model to ignore
- `max_features`: the maximum number of word features to keep. Here, we keep up to 5,000 of the most frequent words after removing stop words.

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    max_features=5000
)

If you call `vectorizer` now, you should be able to review its various parameters:

In [ ]:
vectorizer

Now we can pass texts through the `vectorizer` to get TFIDF results. We'll run it on our "full_text" column values. First we'll use the `skicit-learn` method `.fit_transform`. This looks through the texts and quickly learns the vocubulary. It also applies our settings, `stop_words` and `max_features`. What does this create for us?

In [ ]:
tfidf_matrix = vectorizer.fit_transform(austen_texts["full_text"])

tfidf_matrix

It creates a __matrix__. You can think of this sort of like a dataframe, only a matrix usually contains just the values, without the row and column labels that make a dataframe easier to read. For example, you can look at a column in `tfidf_matrix` like this:

In [ ]:
print(tfidf_matrix[0])

And you can look at a single value like this:

In [ ]:
print(tfidf_matrix[0, 500])

This data structure isn't encoded to be very readable for humans, frankly. It's meant for computers to process in other steps. But basically, these numbers are the weights assigned to the words. To understand them better, we need to do more processing.

To do that, we'll use the `.get_feature_names_out()` method. This gets the list of words that the vectorizer kept based on its weights. We'll use these words as the column names when we turn the TF-IDF matrix into a dataframe.

In [ ]:
terms = vectorizer.get_feature_names_out()

What did that create? Let's check:

In [ ]:
type(terms)

It's an array. An __array__ is a list-like structure. They're designed to hold numbers and speed up calculations. Here, the array stores the vocabulary words that will become the columns of our TF-IDF dataframe.

Now we just need to bring those terms together with our matrix of weights. We'll do that by turning them into a dataframe. To do that, we'll convert the matrix to an array, too, and use the terms as the column values names:

In [ ]:
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=austen_texts["title"], columns=terms)

tfidf_df.shape


Now let's take a look. What have we created?

In [ ]:
tfidf_df.head()

Rows are novels. Columns are terms. Each value is a TF-IDF score. We have the calculations of the frequency of the words in relation to their frequencies in all other texts!

But how can we use this data? We probably don't need to look at every. single. word. We just want to look at the most distinct words based on these results.

We'll probably want to do this several times––once per novel in our data. In that case, let's do it with a function. The function needs to take the tfidf scores from the given title and our chosen number of words. It needs to locate the title in our tfidf_df column, then take the values from that row and arrange them from highest to lowest until it reaches our chosen number of words.

In [ ]:
# defines function, notice how it passes two arguments this time
def top_tfidf_terms(title, top_n=15):
    # creates scores by locating title in tfidf_df
    scores = tfidf_df.loc[title]
    # sorts values
    scores = scores.sort_values(ascending=False)
    # goes to our top_n choice
    scores = scores.head(top_n)

    # returns a dataframe
    return pd.DataFrame({"term": scores.index, "tfidf": scores.values})


Let's try it out on _Emma_.

In [ ]:
top_tfidf_terms("Emma", top_n=20)


It works! That was A LOT of data transformations. But admittedly, TFIDF is more complicated. But let's see the results. First we can visualize the top words in terms of their TF-IDF scores in Emma:

In [ ]:
emma_top_tfidf = top_tfidf_terms("Emma", top_n=20)

emma_top_tfidf.plot(kind="bar", x="term", y="tfidf", legend=False)

plt.title("Top TF-IDF Terms in Emma")
plt.xlabel("Term")
plt.ylabel("TF-IDF")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


What do these results tell you about the work?

Perhaps we can understand more if we compare TF-IDF terms for every novel. This lets us see the corpus-wide comparison more directly than looking at one novel at a time.

To do that, let's compile the top results per novel. In this loop, we'll apply our `top_tfidf_terms` function.

In [ ]:
# our accumulator shell
top_tfidf_rows = []

# for loop that iterates over tfidf_df
for title in tfidf_df.index:
    # applies our function
    top_terms = top_tfidf_terms(title, top_n=10)
    # adds a column called "title"
    top_terms["title"] = title
    # adds the results to our accumulator
    top_tfidf_rows.append(top_terms)

# concatenates each result, ignoring indices
top_tfidf_by_title = pd.concat(top_tfidf_rows, ignore_index=True)
top_tfidf_by_title


Now let's visualize it. I'll skip the full breakdown for now. In our next tutorial, we do a deep dive into visualizations. But to compare tf-idf results across novels, let's look at this:

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize = (8, 5), sharex=True)
axes = axes.flatten()

for ax, title in zip(axes, tfidf_df.index):
    plot_df = top_tfidf_by_title[top_tfidf_by_title["title"] == title]
    plot_df = plot_df.sort_values("tfidf")

    ax.barh(plot_df["term"], plot_df["tfidf"])
    ax.set_title(title)
    ax.set_xlabel("TF-IDF")

fig.suptitle("Top 10 TF-IDF Terms by Jane Austen Novel", fontsize=16)
plt.tight_layout()
plt.show()